# PyTorch Tensor Basics

A targeted recap of tensor mechanics, scoped toward what attention/transformer implementation actually needs: indexing/slicing, memory layout (`view` vs `reshape` vs `.contiguous()`), `transpose`/`permute`, broadcasting, `matmul`, and `softmax`. Not a full generic tensor tutorial -- picking up specific rust, not starting from zero.

## Indexing and slicing

Same slicing syntax as Python lists/NumPy, extended to N dimensions — one slice/index per axis, separated by commas: `t[dim0_slice, dim1_slice, ...]`.

- `t[i]` — index along the first axis (returns a tensor with one fewer dimension).
- `t[i, j]` — index along the first two axes.
- `t[a:b]` — slice along the first axis, `[a, b)`, like a Python list slice.
- `t[a:b, c:d]` — independent slice per axis.
- `t[..., 0]` — `...` (`Ellipsis`) means "all the axes I didn't mention" — fills in as many `:` as needed. Common for grabbing a fixed index on the *last* axis regardless of how many dimensions come before it (e.g. `t[..., 0]` on any-rank tensor grabs index 0 of the last dim).
- `t[-1]` — negative indices count from the end, same as Python.
- `t[::2]` — step slicing works too.
- **Slicing returns a *view***, not a copy — it shares the same underlying memory as the original tensor. Mutating the slice mutates the original.

In [ ]:
import torch
# create a tensor of length 5 with values ranging from zero to 5
a = torch.arange(5)
print(a)
#a = [1,2,3,4]
#a[::2]

tensor([0, 1, 2, 3, 4])


[1, 3]

In [ ]:
# TODO: warm up on indexing/slicing.
# t is shape (2, 3, 4) -- think of it as (batch, seq_len, d_model).
import torch
t = torch.arange(24)#
t = t.reshape(2, 3, 4)
print(t.size())

# 1. Get the first item in the batch -> shape should be (3, 4)
a = t[0, ...]
print(a.size())

# 2. Get the last item along the last dimension, for every batch/seq position -> shape (2, 3)
#    (use Ellipsis)
b = t[..., -1]
print(b.size())

# 3. Get every other position along the seq_len dimension, for the whole batch
b, s, d = t.size()
c= t[:, 0:s:2, -1]
c= t[:, ::2, -1]
print(c.size())

# 4. Take a slice of t, mutate an element in the slice, then print the original t --
#    confirm it changed too (proving slicing returns a view, not a copy)

###### this is genuenly didnt know!!!!! slicing!!!
a = t[..., -1]
print(t[1,1, -1])
a[1, 1] = 3
print(t[1,1, -1])

a = t[0:2, 0:2, :]
print(t[1,1, -1])
a[1, 1] = 1
print(t[1,1, -1])

torch.Size([2, 3, 4])
torch.Size([3, 4])
torch.Size([2, 3])
torch.Size([2, 2])
tensor(19)
tensor(3)
tensor(3)
tensor(1)


## Group 1: Creation & inspection

`torch.tensor()`, `torch.arange()`, `torch.zeros()`/`torch.ones()`/`torch.randn()` (+ `*_like` variants), `torch.eye()`, `.shape`/`.size()`, `.dtype`, `.device`, `.numel()`

In [ ]:
# Group 1: Creation & inspection
# torch.tensor(), torch.arange(), torch.zeros()/torch.ones()/torch.randn() (+ *_like),
# torch.eye(), .shape/.size(), .dtype, .device, .numel()

import torch
import numpy as np

############# torch.tensor##############
a = torch.tensor([1, 2, 3], dtype=torch.float32)
#print(a)

a = np.array([1, 2, 3])
b = torch.tensor(a, dtype=torch.float32)
c = torch.from_numpy(a)
print("this is a:", a)
print("this is b:", b)
print("this is c:", c)
b[0] = 2
print("this is a:", a)
print("this is b:", b)
print("this is c:", c)

c[0] = 2
print("this is a:", a)
print("this is b:", b)
print("this is c:", c)

##### specific tensors######
a = torch.ones([5, 3], dtype=torch.int32)
b = torch.zeros_like(a)
c = torch.eye(5)
#d = torch.randn([2, 3], dtype=float.int32) # cant set the dtype to int32
print(a.size(), a.shape)
print(b.size(), b.shape)
print(a.dtype)
print(d)

##### numel ####
a = torch.ones([5, 3], dtype=torch.int32)
b = torch.zeros_like(a)
c = torch.eye(5)
print(a.numel(), b.numel(), c.numel())


this is a: [1 2 3]
this is b: tensor([1., 2., 3.])
this is c: tensor([1, 2, 3])
this is a: [1 2 3]
this is b: tensor([2., 2., 3.])
this is c: tensor([1, 2, 3])
this is a: [2 2 3]
this is b: tensor([2., 2., 3.])
this is c: tensor([2, 2, 3])
torch.Size([5, 3]) torch.Size([5, 3])
torch.Size([5, 3]) torch.Size([5, 3])
torch.int32
tensor([[1],
        [2],
        [3]])
15 15 25


`torch.tensor()` accepts:

- **Python list/nested list** — `torch.tensor([1, 2, 3])`, or nested for higher rank: `torch.tensor([[1,2],[3,4]])`
- **NumPy array** — `torch.tensor(np_array)` — but this **copies** the data (unlike `torch.from_numpy()`, which shares memory)
- **Scalar** — `torch.tensor(5)` → 0-d tensor
- **Another tensor** — works but PyTorch will warn you to use `.clone().detach()` instead (it copies + loses grad history awkwardly)
- **tuple** — same as list

Does input type matter? Yes, in a couple ways:

- **dtype inference** — all-ints → `int64`, any float present → `float32`. Mixed nested lists must have consistent shape (ragged lists error out).
- **Always copies** — `torch.tensor(...)` always makes a new copy of the data, regardless of source. This is different from `torch.as_tensor()` or `torch.from_numpy()`, which share memory with a NumPy array when possible.

### `torch.tensor()` vs `torch.as_tensor()` vs `torch.from_numpy()`

| | copies? | accepts |
|---|---|---|
| `torch.tensor(x)`      | **always** copies                      | list, tuple, scalar, NumPy array, tensor — anything |
| `torch.as_tensor(x)`   | copies **only if necessary** (dtype/device mismatch, or `x` has no shareable buffer — e.g. a plain Python list) | NumPy array, tensor, list |
| `torch.from_numpy(x)`  | **never** copies, **always** shares memory | NumPy array only |

Example — `as_tensor` sharing vs. copying:

```python
arr = np.array([1, 2, 3])

t = torch.as_tensor(arr)                          # shares memory (same dtype, CPU)
t[0] = 99
print(arr)                                          # [99, 2, 3] -- arr changed too

t2 = torch.as_tensor(arr, dtype=torch.float64)     # copies -- dtype conversion required
t2[0] = 5
print(arr)                                          # unchanged
```

### Random distributions

- **`torch.randn(*sizes)`**              — standard normal (mean 0, std 1)
- **`torch.rand(*sizes)`**                — uniform on `[0, 1)`
- **`torch.randint(low, high, size)`**    — random integers in `[low, high)` -- use this (not `randn`/`rand`) when you need integer output, since sampling a continuous distribution can't directly produce ints.

Rule of thumb: **standard → `randn`, uniform → `rand`, integers → `randint`.**

### `.numel()`

`a.numel()` returns the **total count of elements** in the tensor — the product of all dimension sizes.

```python
a = torch.zeros(3, 4)
a.numel()   # 12  (3 * 4)
```

Different from `.shape`/`.size()`, which give you the *shape* — `.numel()` gives you a single number, the total element count regardless of how it's arranged across dims.

## Group 2: Reshaping & layout

`.view()`, `.reshape()`, `.transpose()`, `.permute()`, `.contiguous()`, `.squeeze()`/`.unsqueeze()`, `.flatten()`, `.expand()`, `.repeat_interleave()` (also how grouped/multi-query attention broadcasts fewer KV heads to match more Q heads)

In [40]:
# Group 2: Reshaping & layout
# .view(), .reshape(), .transpose(), 
# .permute(), .contiguous(), .flatten()

a = torch.tensor([[1, 2], [3, 4], [5, 6]])
print(a.size(), a.shape)

b = a.view(2, 3) # tensor.view(new_shape)
c = a.reshape(2, 3) # tensor.reshape(new_shape)
d = a.permute((1, 0)) #tensor.permute((axis order))
f = a.transpose(1, 0)
print("original", a)
print("view", b)
print("reshape", c)
print("permute", d)
print("trans", f)
g = d.contiguous().view(3, 2) #gives error -> permute and transpose make the tensor non-contiguous 
print("view after permute", g)
h = g.flatten()
print("flatten:", h)

torch.Size([3, 2]) torch.Size([3, 2])
original tensor([[1, 2],
        [3, 4],
        [5, 6]])
view tensor([[1, 2, 3],
        [4, 5, 6]])
reshape tensor([[1, 2, 3],
        [4, 5, 6]])
permute tensor([[1, 3, 5],
        [2, 4, 6]])
trans tensor([[1, 3, 5],
        [2, 4, 6]])
view after permute tensor([[1, 3],
        [5, 2],
        [4, 6]])
flatten: tensor([1, 3, 5, 2, 4, 6])


### Contiguity

A tensor is **contiguous** when its elements sit in memory in the same order as a row-major walk through its shape — moving one step along the last dim moves exactly one step in memory. Freshly created tensors are contiguous by default.

Operations like **`.transpose()`/`.permute()`** don't move any data — they just remap strides (how logical indices map to memory offsets). That breaks the row-major layout without touching the underlying buffer, so the result is **non-contiguous**.

This matters because some ops — like **`.view()`** — require a direct reinterpretation of a contiguous buffer and error out on a non-contiguous tensor:

```python
t = torch.arange(12).reshape(3, 4)
t.is_contiguous()          # True

t2 = t.transpose(0, 1)     # shape (4, 3) -- same memory, different strides
t2.is_contiguous()         # False
t2.view(-1)                 # RuntimeError -- view() needs a contiguous buffer
```

Fix with **`.contiguous()`**, which copies the data into a fresh buffer matching the current shape in row-major order — or just use **`.reshape()`**, which falls back to copying automatically when `.view()` wouldn't work.

Note this is a tensor/NumPy-specific concept — it doesn't apply to Python lists, which aren't backed by a single strided buffer to begin with.

### `.view()` vs `.reshape()`

Same goal (change shape without copying when possible), different guarantees:
- **`.view()`** — requires the tensor to already be contiguous-compatible; **errors** if not.
- **`.reshape()`** — tries to return a view like `.view()`, but silently **falls back to copying** if the tensor isn't contiguous. Safer, less predictable about whether you got a view or a copy.

Neither is in-place: both return a **new tensor object** (new shape/stride metadata) — the original tensor's own `.shape` is never mutated. What *is* shared (when a view is returned) is the underlying **storage**, so mutating an element through the result still mutates the original's data — same idea as slicing. This is different from PyTorch's actual "in-place ops" (trailing-underscore methods like `add_()`), which mutate values directly.

### `.transpose()` / `.permute()` — and why they're not interchangeable with view/reshape

- **`.transpose(dim0, dim1)`** swaps any two axes, works on any rank (e.g. `t.transpose(0, 2)` on a 3D tensor swaps axis 0 and 2, leaves axis 1 alone).
- **`.permute(*dims)`** is the general form — reorders **all** axes at once in one call, needed when more than two axes move.

The key distinction from `.view()`/`.reshape()`: those two only **regroup** elements into new dimensions — the underlying element order in memory never changes. `.transpose()`/`.permute()` **reorder which axis is which** — a real semantic change, not just a relabeling.

```python
t = torch.arange(24).reshape(2, 3, 4)   # (batch, seq, dim)

t.transpose(0, 1)    # shape (3, 2, 4) -- batch/seq genuinely swapped, values correctly reassociated
t.reshape(3, 2, 4)    # shape (3, 2, 4) -- WRONG data: just re-chunks the flat buffer, scrambles batch/seq
```

`.flatten(start_dim, end_dim)` is just a readable special case of `.reshape()` for collapsing a *range* of dims into one — same "regroup" category as reshape, not the "reorder" category as transpose/permute.

In [103]:
#.squeeze()/.unsqueeze(),
# .expand(), expand_as(), .repeat(), .repeat_interleave()
a = torch.tensor([[1], [2], [3]])
b = a.expand(-1, 4)
c = a.repeat(1, 4)
print("this is a:", a.shape)
print("this is b:", b.shape)
print("this is c:", c.shape)
print(b)
print(c)
d = a.flatten().repeat(2)
e = a.flatten().repeat_interleave(2)

#d = a.repeat(1, 2) #Tensor.repeat(*repeats)
e1 = a.repeat(2, 1) #Tensor.repeat(*repeats)
e2 = a.repeat_interleave(2, dim=0) #(repeats, dim)
#print("repeat:", e1)
#print("repeat:", e2)

print("this is d:", d)
print(e)


a = torch.tensor([1, 2, 3])
d = torch.tensor([[1, 2, 3], [1, 2, 3]])
b = a.expand(1, -1)
print("me:", b)
b[0, 1] = 1
print(a) # value changed!!!
c = a.expand_as(d)
print("did you expand?", c)


######## squeeze()/unsqueeze()
a = torch.tensor([1, 2, 3])
b = a.expand(1, -1)
c = b.squeeze()
print("this is c:", c)
d = c.unsqueeze(1) #dim
print("d:", d)

this is a: torch.Size([3, 1])
this is b: torch.Size([3, 4])
this is c: torch.Size([3, 4])
tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])
tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])
this is d: tensor([1, 2, 3, 1, 2, 3])
tensor([1, 1, 2, 2, 3, 3])
me: tensor([[1, 2, 3]])
tensor([1, 1, 3])
did you expand? tensor([[1, 1, 3],
        [1, 1, 3]])
this is c: tensor([1, 2, 3])
d: tensor([[1],
        [2],
        [3]])


### `.expand()` — only stretches singleton (size-1) dimensions

`.expand()` broadcasts by giving a dimension **stride 0** — it doesn't move or duplicate data, it just fakes a larger size by reading the same underlying element repeatedly. That only makes sense starting from a dimension whose real size is **1**.

```python
a = torch.tensor([[1, 2], [3, 4], [5, 6]])   # shape (3, 2)
a.expand(-1, 4)                                # RuntimeError -- dim 1 is already 2, not a singleton, nothing to broadcast
```

If a dim already exists and isn't size 1, `.expand()` has no valid way to stretch it — there's no rule for turning 2 real values into 4. It only works when the source dim is a genuine singleton:

```python
a = torch.tensor([[1], [2], [3]])   # shape (3, 1)
a.expand(-1, 4)                       # shape (3, 4) -- works, dim 1 was 1
```

`.expand()` **can** add brand-new leading dimensions on its own, no `.unsqueeze()` needed. The args you pass are matched to the tensor's existing dims **right to left** — the rightmost args line up with the tensor's real dims, and any extra args on the left (with nothing left to match) become brand-new dims, treated as if they started at size 1:

```python
a = torch.tensor([1, 2, 3])   # shape (3,)
a.expand(5, 3)                  # shape (5, 3) -- works: rightmost arg (3) matches the existing dim, leftmost (5) is new
```

The constraint only bites when an arg is matched up against a dim the tensor **already has** and that dim isn't size 1 (the `(3,2)` example above) — not when there's no existing dim left to match, so a new one gets created.

### `.expand()` / `.repeat()` — minimum number of size args

Both require **at least as many** size arguments as the tensor has dimensions. Extra args (beyond the tensor's rank) get treated as new leading dims and prepended — but *fewer* args than the tensor's rank is invalid, since there'd be no way to know which existing dim a given number refers to:

```python
a = torch.tensor([[1], [2], [3]])   # shape (3, 1) -- 2D
a.repeat(2)                           # RuntimeError -- only 1 repeat-arg given for a 2D tensor
a.repeat(2, 1)                        # shape (6, 1) -- correct: repeat dim0 x2, dim1 x1 (unchanged)
```

### `.expand()` vs `.repeat()`

Same visible output when starting from a singleton dim, different underlying behavior:

```python
a = torch.tensor([[1], [3], [5]])   # shape (3, 1)
a.expand(-1, 4)                       # tensor([[1,1,1,1],[3,3,3,3],[5,5,5,5]])
a.repeat(1, 4)                        # tensor([[1,1,1,1],[3,3,3,3],[5,5,5,5]])  -- same values
```

| | `.expand()` | `.repeat()` |
|---|---|---|
| memory | **view** — no new memory, stride 0 on the broadcast dim | **copy** — real new memory, `n`x the size |
| writes | mutating one "copy" mutates **all** of them (same memory cell) | each copy is independent |
| requires | source dim must be size **1** (to stretch an existing dim) | works on any existing dim, no singleton needed |
| cost | free | allocates + copies |

Rule of thumb: `.expand()` for cheap, read-only broadcasting (e.g. lining up shapes for elementwise math); `.repeat()` when you need a real, independently-writable, tiled copy.

### `.repeat()` vs `.repeat_interleave()`

Different tiling order — `.repeat()` copies the **whole block**, `.repeat_interleave()` duplicates **each element in place** before moving to the next.

```python
a = torch.tensor([1, 2, 3])

a.repeat(2)              # tensor([1, 2, 3, 1, 2, 3])       -- whole sequence tiled
a.repeat_interleave(2)   # tensor([1, 1, 2, 2, 3, 3])       -- each element duplicated in place
```

Maps onto NumPy, but the naming doesn't line up 1:1 — worth memorizing explicitly:
- `torch.repeat()` ≈ `np.tile()`
- `torch.repeat_interleave()` ≈ `np.repeat()`

This is exactly the mechanism behind grouped/multi-query attention broadcasting fewer KV heads to match more Q heads: each KV head's block needs to repeat *contiguously* so it lines up with all the Q heads assigned to it — that's `repeat_interleave`, not `repeat`, which would tile the whole KV sequence and misalign the grouping.

### Does the `.view()` contiguity restriction also apply to `.reshape()`?

Yes, the underlying constraint is identical — `.reshape()` checks the exact same "is this contiguous-compatible?" condition internally. The difference is only in what happens when that check fails: `.view()` raises a `RuntimeError`; `.reshape()` silently does `.contiguous().view(...)` for you and returns a copy instead of erroring.

### `.squeeze()` / `.unsqueeze()`

`.squeeze()` removes size-1 dims. `.unsqueeze(dim)` inserts a new size-1 dim at `dim`.

```python
a = torch.tensor([[1, 2, 3]])   # shape (1, 3)
a.squeeze()                       # shape (3,) -- removes the size-1 dim
a.squeeze(0)                      # shape (3,) -- remove only dim 0, safer/explicit

b = torch.tensor([1, 2, 3])     # shape (3,)
b.unsqueeze(0)                    # shape (1, 3) -- new dim at front
b.unsqueeze(1)                    # shape (3, 1) -- new dim at end
```

- `.squeeze()` with no arg removes **all** size-1 dims at once — can accidentally squeeze one you meant to keep (e.g. a batch dim that happens to be 1). Passing a specific `dim` is safer.
- `.squeeze(dim)` on a dim that isn't size 1 does nothing — no error, just a no-op.
- Both are views (share memory), no copy.
- `.unsqueeze()` is exactly what gives `.expand()` a real dim to attach to when you need to broadcast into an axis the tensor doesn't have.
- **`.unsqueeze(dim)` only takes a single `int`, not a list** — `c.unsqueeze([1, 2])` errors: `TypeError: unsqueeze() argument 'dim' (position 1) must be int, not list`. Unlike some ops that accept a list of dims, `.unsqueeze()` only ever inserts **one** new dim per call — to add two new dims, call it twice: `c.unsqueeze(1).unsqueeze(2)`.

## Group 3: Copying vs. views

`.clone()` — contrast with slicing/`.view()`, which share memory with the original.

In [ ]:
# Group 3: Copying vs. views
# .clone() -- contrast with slicing/.view(), which share memory with the original
a = torch.tensor([1, 2, 3])
b = a.clone()
b[0] = 0
print(a, b)

a = torch.tensor([1, 2, 3])
b = a.view(-1)
b[0] = 0
print(a, b)

a = torch.tensor([1, 2, 3])
b = a.view(-1).clone()
b[0] = 0
print(a, b)

a = torch.tensor([1, 2, 3])
b = a.clone().view(-1) ## recommended way of doing it
b[0] = 0
print(a, b)

tensor([1, 2, 3]) tensor([0, 2, 3])
tensor([0, 2, 3]) tensor([0, 2, 3])
tensor([1, 2, 3]) tensor([0, 2, 3])
tensor([1, 2, 3]) tensor([0, 2, 3])


## Group 4: Combining & splitting

`torch.cat()` (also how a KV cache appends new keys/values along the sequence dim each generation step), `torch.stack()`, `.split()`, `.chunk()`

In [144]:
# Group 4: Combining & splitting
# torch.cat(), torch.stack(), .split(), .chunk()

a = torch.tensor([1, 2, 3, 7, 8])
b = torch.tensor([4, 5, 6, 9, 10])

c = torch.cat([a, b], dim=0)
#c = torch.cat([a, b], dim=1) ## returns error

c = torch.stack([a, b], dim=0)
d = torch.stack([a, b], dim=1)
#print(c)
#print(d)

##### .split() #####
e = d.split(2)
m = d.split([2, 1, 2])             
print(m[0].size(), m[1].size(), m[2].size())

a = torch.randn([4, 5, 6])
b = a.split([2, 3, 1], dim=-1) # split can do along one dimension only!!
print(b[0].size(), b[1].size(), b[2].size())

##### .chunk() #####
x = torch.arange(10)
y = x.chunk(3)   # number of pieces given, sizes computed automatically
print([p.size() for p in y], y)
y = x.tensor_split(3)   # number of pieces given, sizes computed automatically
print([p.size() for p in y], y)

print("+++++++")
a = torch.randn([4, 5, 6]) #chunks: int, dim: int = 0
y = a.chunk(3, 1)
print([p.size() for p in y])

a = torch.randn([4, 4, 6]) 
y = a.chunk(3, 1) ### interestingly enough doesnt do 3 chunks
print([p.size() for p in y])

torch.Size([2, 2]) torch.Size([1, 2]) torch.Size([2, 2])
torch.Size([4, 5, 2]) torch.Size([4, 5, 3]) torch.Size([4, 5, 1])
[torch.Size([4]), torch.Size([4]), torch.Size([2])] (tensor([0, 1, 2, 3]), tensor([4, 5, 6, 7]), tensor([8, 9]))
[torch.Size([4]), torch.Size([3]), torch.Size([3])] (tensor([0, 1, 2, 3]), tensor([4, 5, 6]), tensor([7, 8, 9]))
+++++++
[torch.Size([4, 2, 6]), torch.Size([4, 2, 6]), torch.Size([4, 1, 6])]
[torch.Size([4, 2, 6]), torch.Size([4, 2, 6])]


### `torch.cat()` vs `torch.stack()`

**`torch.cat(tensors, dim)`** — joins tensors along an **existing** dimension; doesn't add a new one. All tensors must match in every dim except the one being joined.

```python
a = torch.tensor([[1, 2]])   # shape (1, 2)
b = torch.tensor([[3, 4]])   # shape (1, 2)
torch.cat([a, b], dim=0)       # shape (2, 2) -- joined along dim 0, which already existed
# tensor([[1, 2], [3, 4]])
```

**`torch.stack(tensors, dim)`** — joins tensors along a **brand new** dimension. All tensors must have the exact same shape.

```python
a = torch.tensor([1, 2])   # shape (2,)
b = torch.tensor([3, 4])   # shape (2,)
torch.stack([a, b], dim=0)   # shape (2, 2) -- new dim inserted at 0
# tensor([[1, 2], [3, 4]])
```

Same "existing dim" vs "new dim" split seen with `.expand()`/`.repeat()`: `cat` grows a dim that's already there, `stack` creates a dim that wasn't.

### `.split()` vs `.chunk()`

Both break a tensor into pieces along a dim — the reverse direction of `cat`/`stack`.

**`.split(size, dim)`** — pieces of a given **size** (or list of sizes).

```python
a = torch.arange(6)   # tensor([0,1,2,3,4,5])
a.split(2)               # (tensor([0,1]), tensor([2,3]), tensor([4,5])) -- pieces of size 2
```

**`.chunk(n, dim)`** — a given **number** of pieces, sizes computed automatically (as evenly as possible).

```python
a = torch.arange(6)
a.chunk(3)               # (tensor([0,1]), tensor([2,3]), tensor([4,5])) -- 3 pieces, sizes inferred
```

Pick based on which constraint you actually have: exact size per piece → `.split()`; exact piece count → `.chunk()`.

### `.split()` signature — `split_size_or_sections` can be int or list

`torch.split(tensor, split_size_or_sections, dim=0)`:

- **int** — every piece is that size, except possibly the last, which is smaller if the dim doesn't divide evenly.
- **list of ints** — exact size for each piece, in order. The list's values must **sum to the size of `dim`**, or it errors.

```python
a = torch.arange(7)          # size 7
a.split(3)                     # (tensor([0,1,2]), tensor([3,4,5]), tensor([6]))  -- last piece smaller
a.split([2, 4, 1])             # (tensor([0,1]), tensor([2,3,4,5]), tensor([6]))  -- 2+4+1 = 7, matches
a.split([2, 4, 2])             # RuntimeError -- 2+4+2 = 8 ≠ 7
```

### `torch.tensor_split()` — not `.split()`, and not quite `.chunk()` either

`torch.tensor_split(input, indices_or_sections, dim=0)` is a third splitting function, distinct from both:

- **int `n`** — similar to `.chunk(n)` in that it means "number of pieces," but the guarantees differ: `.chunk(n)` computes piece size as `ceil(size/n)`, so it front-loads size into the first pieces and can return **fewer than `n`** pieces if the dim is too small. `.tensor_split(n)` always returns **exactly `n`** pieces, sizes balanced as evenly as possible (differ by at most 1).
- **list** — for `.split()`, the list means **sizes** (must sum to the dim's total). For `.tensor_split()`, the list means **cut points (indices)** — doesn't need to sum to anything, just marks where to slice.

```python
a = torch.arange(7)
a.chunk(3)                  # sizes: 3, 3, 1   -- ceil(7/3)=3 per piece, last one smaller
a.tensor_split(3)            # sizes: 3, 2, 2   -- exactly 3 pieces, balanced
a.tensor_split([2, 5])       # t[:2], t[2:5], t[5:]  -- list = cut indices, not sizes
```

## Group 5: Core math

`@`/`torch.matmul()`, `torch.bmm()`, `torch.einsum()`, `torch.softmax()`, `F.log_softmax()`, elementwise `+ - * /`, `torch.exp()`, `torch.log()`, `torch.sqrt()`, `torch.rsqrt()`, `torch.cos()`/`torch.sin()` (rotary embeddings), `torch.sum()`/`.mean()` (with `dim=`), `torch.max()`/`.argmax()`, `torch.topk()`

In [ ]:
# Group 5: Core math
# @ / torch.matmul(), torch.bmm(), torch.einsum(), torch.softmax(), F.log_softmax(),
# elementwise + - * /, torch.exp(), torch.log(), torch.sqrt(), torch.rsqrt(),
# torch.cos()/torch.sin() (rotary embeddings), torch.sum()/.mean() (dim=),
# torch.max()/.argmax(), torch.topk()


## Group 6: Masking & selecting

Boolean indexing (`t[mask]`), `torch.where()`, `.masked_fill()`, `torch.triu()`/`torch.tril()` (causal mask; a banded combination of the two also builds a sliding-window mask), `torch.gather()`

In [ ]:
# Group 6: Masking & selecting
# boolean indexing (t[mask]), torch.where(), .masked_fill(), torch.triu()/torch.tril()
# (causal mask; banded triu+tril also builds a sliding-window mask), torch.gather()


## Group 7: LLM-specific layers/ops

`nn.Embedding`, `F.layer_norm()`/`nn.LayerNorm`, `F.gelu()`, `F.cross_entropy()`, `F.kl_div()` (policy-vs-reference penalty in PPO-style RLHF), `F.scaled_dot_product_attention()` (dispatches to a flash-attention kernel automatically when conditions allow), `F.pad()`

In [ ]:
# Group 7: LLM-specific layers/ops
# nn.Embedding, F.layer_norm()/nn.LayerNorm, F.gelu(), F.cross_entropy(),
# F.kl_div() (RLHF policy-vs-reference penalty),
# F.scaled_dot_product_attention() (auto-dispatches to flash attention), F.pad()

import torch.nn as nn
import torch.nn.functional as F

## Group 8: Sampling / generation

`torch.multinomial()`, `torch.topk()` (cross-ref Group 5), `.argmax()` for greedy decoding

In [ ]:
# Group 8: Sampling / generation
# torch.multinomial(), torch.topk() (cross-ref Group 5), .argmax() for greedy decoding


## Group 9: Device, dtype & inference mode

`.to(device)`/`.to(dtype)`, `.float()`/`.half()`, `.item()`, `torch.no_grad()`, `.detach()` (stop gradient flow into a frozen/reference or reward model — related to `no_grad()` but detaches a specific tensor rather than a whole block), `torch.clamp()`

In [ ]:
# Group 9: Device, dtype & inference mode
# .to(device)/.to(dtype), .float()/.half(), .item(), torch.no_grad(), .detach(),
# torch.clamp()


## Group 10: Debugging/testing

`torch.allclose()`, `torch.equal()`

In [ ]:
# Group 10: Debugging/testing
# torch.allclose(), torch.equal()
